# Лабораторная работа №1: Уменьшение размерности данных

## Цель работы
Освоить практические навыки работы с методами уменьшения размерности данных, такими как PCA и t-SNE, а также интерпретации их результатов.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Настройка отображения
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

print("Библиотеки загружены успешно")


## 1. Сбор и предобработка данных


In [ ]:
# Загрузка данных
df = pd.read_csv('music_genre.csv')
print(f"Размер исходного датасета: {df.shape}")
print(f"\nКолонки: {df.columns.tolist()}")
print(f"\nПервые строки:")
df.head()


In [ ]:
# Проверка типов данных и пропущенных значений
print("Информация о датасете:")
print(df.info())
print("\nПропущенные значения:")
print(df.isnull().sum())
print("\nУникальные значения в категориальных колонках:")
print(f"key: {df['key'].unique()}")
print(f"mode: {df['mode'].unique()}")
print(f"music_genre: {df['music_genre'].unique()}")


In [ ]:
# a) Замена нечисловых данных на числовые
# Создаем копию датасета для работы
df_processed = df.copy()

# Создаем шкалу соответствия для key
key_mapping = {
    'C': 0, 'C#': 1, 'D': 2, 'D#': 3, 'E': 4, 'F': 5,
    'F#': 6, 'G': 7, 'G#': 8, 'A': 9, 'A#': 10, 'B': 11
}
df_processed['key'] = df_processed['key'].map(key_mapping)

# Создаем шкалу соответствия для mode
mode_mapping = {'Major': 1, 'Minor': 0}
df_processed['mode'] = df_processed['mode'].map(mode_mapping)

# Сохраняем метки жанров для визуализации
genre_labels = df_processed['music_genre'].copy()
genre_mapping = {genre: idx for idx, genre in enumerate(genre_labels.unique())}
df_processed['music_genre_encoded'] = df_processed['music_genre'].map(genre_mapping)

print("Преобразование категориальных переменных завершено")
print(f"\nМаппинг key: {key_mapping}")
print(f"Маппинг mode: {mode_mapping}")
print(f"\nКоличество жанров: {len(genre_labels.unique())}")


In [ ]:
# Выбираем числовые колонки для анализа (исключаем текстовые и ID)
numeric_cols = ['popularity', 'acousticness', 'danceability', 'duration_ms', 
                'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 
                'mode', 'speechiness', 'tempo', 'valence']

# Проверяем наличие всех колонок
available_cols = [col for col in numeric_cols if col in df_processed.columns]
print(f"Доступные числовые колонки: {available_cols}")

# Обработка пропущенных значений и некорректных данных
df_numeric = df_processed[available_cols].copy()

# Заменяем '?' и другие некорректные значения на NaN
for col in df_numeric.columns:
    if df_numeric[col].dtype == 'object':
        df_numeric[col] = pd.to_numeric(df_numeric[col], errors='coerce')

# Удаляем строки с пропущенными значениями
df_numeric = df_numeric.dropna()
print(f"\nРазмер после удаления пропущенных значений: {df_numeric.shape}")

# b) Удаление переменных с нулевой дисперсией
variances = df_numeric.var()
zero_var_cols = variances[variances == 0].index.tolist()
print(f"\nКолонки с нулевой дисперсией: {zero_var_cols}")

if zero_var_cols:
    df_numeric = df_numeric.drop(columns=zero_var_cols)
    print(f"Колонки удалены. Новый размер: {df_numeric.shape}")

# c) Оставляем 10-12 параметров
# Выбираем наиболее информативные параметры
selected_cols = df_numeric.columns.tolist()[:12]  # Берем первые 12 доступных
if len(selected_cols) > 12:
    selected_cols = selected_cols[:12]
elif len(selected_cols) < 10:
    selected_cols = df_numeric.columns.tolist()[:10]

df_final = df_numeric[selected_cols].copy()
print(f"\nВыбранные параметры ({len(selected_cols)}): {selected_cols}")
print(f"Финальный размер данных: {df_final.shape}")

# Обновляем genre_labels для соответствия индексам
genre_labels = genre_labels.loc[df_final.index]


In [ ]:
# c) Построение корреляционной матрицы
correlation_matrix = df_final.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Корреляционная матрица параметров музыки', fontsize=16, pad=20)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Анализ сильных корреляций
print("\nСильные корреляции (|r| > 0.5):")
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_val = correlation_matrix.iloc[i, j]
        if abs(corr_val) > 0.5:
            print(f"{correlation_matrix.columns[i]} - {correlation_matrix.columns[j]}: {corr_val:.3f}")


In [ ]:
# d) Построение графика каменистой осыпи (Scree Plot)
# Для этого нужно сначала применить PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_final)

pca_temp = PCA()
pca_temp.fit(X_scaled)

# График каменистой осыпи
plt.figure(figsize=(10, 6))
components = range(1, len(pca_temp.explained_variance_) + 1)
plt.plot(components, pca_temp.explained_variance_, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Номер главной компоненты', fontsize=12)
plt.ylabel('Собственное значение (Eigenvalue)', fontsize=12)
plt.title('График каменистой осыпи (Scree Plot)', fontsize=14, pad=20)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('scree_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print("График каменистой осыпи построен")


## 2. Реализация и применение PCA


In [ ]:
# a) Реализация PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

print("PCA применен успешно")
print(f"Количество компонент: {pca.n_components_}")
print(f"Размерность исходных данных: {X_scaled.shape}")
print(f"Размерность после PCA: {X_pca.shape}")


In [ ]:
# b) Вычисление объясненной дисперсии для каждой компоненты
explained_variance = pca.explained_variance_ratio_

print("Объясненная дисперсия для каждой компоненты:")
for i, var in enumerate(explained_variance, 1):
    print(f"Компонента {i}: {var:.4f} ({var*100:.2f}%)")

# c) Вычисление кумулятивной объясненной дисперсии
cumulative_variance = np.cumsum(explained_variance)

print("\nКумулятивная объясненная дисперсия:")
for n in [1, 2, 3, 5, 10]:
    if n <= len(cumulative_variance):
        print(f"Первые {n} компонент: {cumulative_variance[n-1]:.4f} ({cumulative_variance[n-1]*100:.2f}%)")


In [ ]:
# График объясненной дисперсии
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# График объясненной дисперсии для каждой компоненты
components = range(1, len(explained_variance) + 1)
ax1.bar(components, explained_variance, alpha=0.7, color='steelblue')
ax1.plot(components, explained_variance, 'ro-', linewidth=2, markersize=6)
ax1.set_xlabel('Номер главной компоненты', fontsize=12)
ax1.set_ylabel('Объясненная дисперсия', fontsize=12)
ax1.set_title('Объясненная дисперсия по компонентам', fontsize=14)
ax1.grid(True, alpha=0.3)

# График кумулятивной объясненной дисперсии
ax2.plot(components, cumulative_variance, 'go-', linewidth=2, markersize=6)
ax2.axhline(y=0.8, color='r', linestyle='--', label='80% дисперсии')
ax2.axhline(y=0.9, color='orange', linestyle='--', label='90% дисперсии')
ax2.set_xlabel('Количество компонент', fontsize=12)
ax2.set_ylabel('Кумулятивная объясненная дисперсия', fontsize=12)
ax2.set_title('Кумулятивная объясненная дисперсия', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pca_variance.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Интерпретация главных компонент
print("Интерпретация главных компонент:")
print("\nПервая главная компонента (PC1):")
pc1_loadings = pd.Series(pca.components_[0], index=df_final.columns)
pc1_sorted = pc1_loadings.abs().sort_values(ascending=False)
print(pc1_sorted.head(5))

print("\nВторая главная компонента (PC2):")
pc2_loadings = pd.Series(pca.components_[1], index=df_final.columns)
pc2_sorted = pc2_loadings.abs().sort_values(ascending=False)
print(pc2_sorted.head(5))

# Визуализация нагрузок компонент
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

for i in range(4):
    ax = axes[i // 2, i % 2]
    loadings = pd.Series(pca.components_[i], index=df_final.columns)
    loadings_sorted = loadings.sort_values()
    ax.barh(range(len(loadings_sorted)), loadings_sorted.values, color='steelblue')
    ax.set_yticks(range(len(loadings_sorted)))
    ax.set_yticklabels(loadings_sorted.index)
    ax.set_xlabel('Нагрузка компоненты', fontsize=11)
    ax.set_title(f'PC{i+1} (Объясненная дисперсия: {explained_variance[i]:.2%})', fontsize=12)
    ax.grid(True, alpha=0.3, axis='x')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=1)

plt.tight_layout()
plt.savefig('pca_loadings.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# d) Построение проекции данных на новое пространство (первые 2 компоненты)
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

# Визуализация в пространстве первых двух главных компонент
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], 
                     c=genre_labels.map(genre_mapping), 
                     cmap='tab20', alpha=0.6, s=20)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} дисперсии)', fontsize=12)
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} дисперсии)', fontsize=12)
plt.title('Проекция данных на пространство первых двух главных компонент', fontsize=14, pad=20)
plt.colorbar(scatter, label='Жанр музыки')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pca_2d_projection.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Объясненная дисперсия первыми двумя компонентами: {pca_2d.explained_variance_ratio_.sum():.2%}")


## 3. Реализация и применение t-SNE


In [ ]:
# a) Реализация t-SNE с разными значениями перплексии
# Используем подвыборку для ускорения вычислений (t-SNE медленный на больших данных)
sample_size = min(5000, len(X_scaled))
np.random.seed(42)
sample_indices = np.random.choice(len(X_scaled), sample_size, replace=False)
X_sample = X_scaled[sample_indices]
genre_labels_sample = genre_labels.iloc[sample_indices]

print(f"Размер выборки для t-SNE: {sample_size}")

# Применяем t-SNE с разными значениями перплексии
perplexities = [5, 30, 50, 100]
tsne_results = {}

for perp in perplexities:
    print(f"\nВычисление t-SNE с перплексией {perp}...")
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42, n_iter=1000)
    X_tsne = tsne.fit_transform(X_sample)
    tsne_results[perp] = X_tsne
    print(f"Завершено для перплексии {perp}")

print("\nВсе вычисления t-SNE завершены")


In [ ]:
# b) Визуализация результатов t-SNE для разных значений перплексии
fig, axes = plt.subplots(2, 2, figsize=(16, 16))
axes = axes.flatten()

for idx, perp in enumerate(perplexities):
    ax = axes[idx]
    X_tsne = tsne_results[perp]
    scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], 
                        c=genre_labels_sample.map(genre_mapping),
                        cmap='tab20', alpha=0.6, s=20)
    ax.set_xlabel('t-SNE 1', fontsize=11)
    ax.set_ylabel('t-SNE 2', fontsize=11)
    ax.set_title(f't-SNE с перплексией {perp}', fontsize=12, pad=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('tsne_perplexities.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Детальная визуализация для оптимальной перплексии (30)
plt.figure(figsize=(12, 8))
X_tsne_optimal = tsne_results[30]
scatter = plt.scatter(X_tsne_optimal[:, 0], X_tsne_optimal[:, 1],
                     c=genre_labels_sample.map(genre_mapping),
                     cmap='tab20', alpha=0.7, s=30, edgecolors='black', linewidth=0.5)
plt.xlabel('t-SNE 1', fontsize=12)
plt.ylabel('t-SNE 2', fontsize=12)
plt.title('t-SNE визуализация (перплексия = 30)', fontsize=14, pad=20)
plt.colorbar(scatter, label='Жанр музыки')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('tsne_optimal.png', dpi=300, bbox_inches='tight')
plt.show()


## 4. Сравнительный анализ PCA и t-SNE


In [ ]:
# Сравнительная визуализация PCA и t-SNE
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# PCA визуализация (на той же выборке)
X_pca_sample = pca_2d.transform(X_sample)

# PCA
scatter1 = axes[0].scatter(X_pca_sample[:, 0], X_pca_sample[:, 1],
                          c=genre_labels_sample.map(genre_mapping),
                          cmap='tab20', alpha=0.6, s=20)
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%})', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%})', fontsize=12)
axes[0].set_title('PCA проекция', fontsize=14)
axes[0].grid(True, alpha=0.3)

# t-SNE
scatter2 = axes[1].scatter(X_tsne_optimal[:, 0], X_tsne_optimal[:, 1],
                          c=genre_labels_sample.map(genre_mapping),
                          cmap='tab20', alpha=0.6, s=20)
axes[1].set_xlabel('t-SNE 1', fontsize=12)
axes[1].set_ylabel('t-SNE 2', fontsize=12)
axes[1].set_title('t-SNE проекция (перплексия=30)', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pca_vs_tsne.png', dpi=300, bbox_inches='tight')
plt.show()

print("Сравнительный анализ завершен")


In [ ]:
# Анализ кластеризации по жанрам
print("Анализ разделимости жанров:")
print(f"\nКоличество уникальных жанров: {len(genre_labels.unique())}")
print(f"Распределение по жанрам:")
print(genre_labels.value_counts().head(10))


2
